<a href="https://colab.research.google.com/github/Josue-Martinez10/Dev2/blob/Projectss/Marketing_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
pip install langgraph langchain langchain_openai pydantic


In [20]:

import os
import operator
from typing import TypedDict, Annotated, List

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

from google.colab import userdata

api_key = userdata.get("OpenAi")
os.environ["OPENAI_API_KEY"] = api_key

class AgentState(TypedDict):
    """
    Represents the state of our ad optimization agent.
    - messages: A list of messages/history.
    - campaign_data: The current (simulated) ad campaign metrics.
    - next_action: The recommended action from the analysis.
    """
    messages: Annotated[List[BaseMessage], operator.add]
    campaign_data: dict
    next_action: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [21]:
@tool
def adjust_bid(campaign_id: str, new_bid_amount: float) -> str:
    """Adjusts the bid for a specific ad campaign to the new amount."""
    if new_bid_amount > 8.00:
        return f"Bid for Campaign '{campaign_id}' adjusted to ${new_bid_amount:.2f}. (Simulated: High bid warning!)"
    return f"Bid for Campaign '{campaign_id}' adjusted to ${new_bid_amount:.2f} successfully."

@tool
def pause_ad_group(ad_group_id: str) -> str:
    """Pauses an underperforming ad group by its ID."""
    return f"Ad Group '{ad_group_id}' paused successfully due to poor performance."

@tool
def request_new_creatives(campaign_id: str) -> str:
    """Requests new ad creative assets from the creative team."""
    return f"New creative request submitted for Campaign '{campaign_id}'. Awaiting design feedback."

ad_optimization_tools = [adjust_bid, pause_ad_group, request_new_creatives]

In [22]:
def fetch_campaign_data(state: AgentState) -> dict:
    """Simulates fetching the latest campaign data."""
    print("--- FETCHING DATA ---")


    simulated_data = {
        "Campaign-2024-Q4": {
            "Budget": 1000, "Spend": 850, "Impressions": 50000,
            "Clicks": 500, "Conversions": 5, "CPA": 170.00, "Target_CPA": 50.00,
            "Ad_Groups": {
                "AG-101": {"Status": "Active", "CPA": 25.00, "Bid": 2.50},
                "AG-102": {"Status": "Active", "CPA": 450.00, "Bid": 3.00} # Poor performer
            }
        }
    }

    analysis_prompt = (
        "Ad Campaign Data for Optimization:\n"
        f"{simulated_data}\n\n"
        "**Optimization Goal:** Reduce overall CPA (Cost Per Acquisition) to be closer to the Target CPA ($50.00) "
        "and maximize conversions. Analyze the data and recommend the *single best action* "
        "using one of the provided tools (adjust_bid, pause_ad_group, request_new_creatives). "
        "Your final output should be ONLY the recommended action as a message for the user, "
        "or a clear instruction for the next internal step."
    )

    new_messages = [HumanMessage(content=analysis_prompt)]

    return {"messages": new_messages, "campaign_data": simulated_data}

def agent_reasoning(state: AgentState) -> dict:
    """The LLM reasons and decides the next step (tool call or final answer)."""
    print("--- AGENT REASONING ---")

    llm_with_tools = llm.bind_tools(ad_optimization_tools)

    system_prompt = (
        "You are an expert Ad Campaign Optimization Agent. "
        "Your goal is to analyze the provided campaign data and decide the optimal next action. "
        "You must use a tool if an optimization is possible. "
        "If no tool is needed or you have executed a tool, provide a final, concise update."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}")
    ])

    chain = prompt | llm_with_tools
    response = chain.invoke(state)

    if response.tool_calls:
        return {"messages": [response], "next_action": "call_tool"}
    else:

        return {"messages": [response], "next_action": "FINISH"}


def execute_tools(state: AgentState) -> dict:
    """Executes the tool call(s) decided by the agent_reasoning step."""
    print("--- EXECUTING TOOL ---")

    tool_calls = state["messages"][-1].tool_calls
    tool_results = []

    for call in tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        tool_call_id = call["id"]

        tool_to_run = next(
            (t for t in ad_optimization_tools if t.name == tool_name), None
        )

        if tool_to_run:
            try:
                result = tool_to_run.invoke(tool_args)
                tool_results.append({
                    "type": "tool_output",
                    "tool_call_id": tool_call_id,
                    "content": result
                })
            except Exception as e:
                 tool_results.append({
                    "type": "tool_output",
                    "tool_call_id": tool_call_id,
                    "content": f"Error executing tool {tool_name}: {e}"
                })



    tool_messages = [HumanMessage(content=str(r), name="tool_execution") for r in tool_results]

    return {"messages": tool_messages}

def decide_next_step(state: AgentState) -> str:
    """Conditional edge: decides whether to continue analysis or finish."""
    print(f"--- DECIDING NEXT STEP ---")
    print(f"state['next_action']: {state.get('next_action')}")
    next_action = state.get("next_action")

    if next_action == "call_tool":
        print("Returning 'call_tool'")
        return "call_tool"
    elif next_action == "FINISH":
        print("Returning 'end'")
        return "end"
    else:
        print("Returning 'agent_reasoning'")
        return "agent_reasoning"

In [23]:
from langchain_core.messages import ToolMessage

def fetch_campaign_data(state: AgentState) -> dict:
    """Simulates fetching the latest campaign data."""
    print("--- FETCHING DATA ---")

    simulated_data = {
        "Campaign-2024-Q4": {
            "Budget": 10000, "Spend": 1200, "Impressions": 69000,
            "Clicks": 650, "Conversions": 8, "CPA": 200.00, "Target_CPA": 89.00,
            "Ad_Groups": {
                "Google Ads": {"Status": "Active", "CPA": 35.00, "Bid": 6.00},
                "SJSU Union Board": {"Status": "Active", "CPA": 200, "Bid": 5.00},
                "Tiktok": {"Status": "Active", "CPA":280, "Bid": 9.00}
            }
        }
    }

    analysis_prompt = (
        "Ad Campaign Data for Optimization:\n"
        f"{simulated_data}\n\n"
        "**Optimization Goal:** Reduce overall CPA (Cost Per Acquisition) to be closer to the Target CPA ($89.00) "
        "and maximize conversions. Analyze the data and recommend the *single best action* "
        "using one of the provided tools (adjust_bid, pause_ad_group, request_new_creatives). "
        "Your final output should be ONLY the recommended action as a message for the user, "
        "or a clear instruction for the next internal step."
    )


    new_messages = [HumanMessage(content=analysis_prompt)]

    return {"messages": new_messages, "campaign_data": simulated_data}

def agent_reasoning(state: AgentState) -> dict:
    """The LLM reasons and decides the next step (tool call or final answer)."""
    print("--- AGENT REASONING ---")


    llm_with_tools = llm.bind_tools(ad_optimization_tools)

    system_prompt = (
        "You are an expert Ad Campaign Optimization Agent. "
        "Your goal is to analyze the provided campaign data and decide the optimal next action. "
        "You must use a tool if an optimization is possible. "
        "If no tool is needed or you have executed a tool, provide a final, concise update."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}")
    ])

    chain = prompt | llm_with_tools
    response = chain.invoke(state)

    if response.tool_calls:
        return {"messages": [response], "next_action": "call_tool"}
    else:
        return {"messages": [response], "next_action": "FINISH"}


def execute_tools(state: AgentState) -> dict:
    """Executes the tool call(s) decided by the agent_reasoning step."""
    print("--- EXECUTING TOOL ---")

    tool_calls = state["messages"][-1].tool_calls
    tool_results = []

    for call in tool_calls:
        tool_name = call["name"]
        tool_args = call["args"]
        tool_call_id = call["id"]

        tool_to_run = next(
            (t for t in ad_optimization_tools if t.name == tool_name), None
        )

        if tool_to_run:
            try:
                result = tool_to_run.invoke(tool_args)
                tool_results.append(ToolMessage(
                    content=result,
                    tool_call_id=tool_call_id
                ))
            except Exception as e:

                 tool_results.append(ToolMessage(
                    content=f"Error executing tool {tool_name}: {e}",
                    tool_call_id=tool_call_id
                ))


    return {"messages": tool_results}

def decide_next_step(state: AgentState) -> str:
    """Conditional edge: decides whether to continue analysis or finish."""
    print(f"--- DECIDING NEXT STEP ---")
    print(f"state['next_action']: {state.get('next_action')}")

    next_action = state.get("next_action")

    if next_action == "call_tool":
        print("Returning 'call_tool'")
        return "call_tool"
    elif next_action == "FINISH":
        print("Returning 'end'")
        return "end"
    else:
        print("Returning 'agent_reasoning'")
        return "agent_reasoning"

In [24]:
workflow = StateGraph(AgentState)


workflow.add_node("fetch_data", fetch_campaign_data)
workflow.add_node("agent_reasoning", agent_reasoning)
workflow.add_node("execute_tools", execute_tools)


workflow.set_entry_point("fetch_data")


workflow.add_edge("fetch_data", "agent_reasoning")

workflow.add_conditional_edges(
    "agent_reasoning",
    decide_next_step,
    {"call_tool": "execute_tools", "end": END}
)

workflow.add_edge("execute_tools", "agent_reasoning")

app = workflow.compile()


print("--- STARTING AD OPTIMIZATION AGENT RUN ---")


final_state = app.invoke(
    {"messages": [], "campaign_data": {}, "next_action": "start"},
    config={"recursion_limit": 50}
)


final_message = final_state["messages"][-1].content
print("\n" + "="*50)
print("AGENT FINAL RECOMMENDATION & SUMMARY:")
print(final_message)
print("="*50)



--- STARTING AD OPTIMIZATION AGENT RUN ---
--- FETCHING DATA ---
--- AGENT REASONING ---
--- DECIDING NEXT STEP ---
state['next_action']: call_tool
Returning 'call_tool'
--- EXECUTING TOOL ---
--- AGENT REASONING ---
--- DECIDING NEXT STEP ---
state['next_action']: FINISH
Returning 'end'

AGENT FINAL RECOMMENDATION & SUMMARY:
The underperforming ad groups 'SJSU Union Board' and 'Tiktok' have been successfully paused to optimize the campaign.
